### Imports and setup

In [0]:
%pip install sqlalchemy psycopg2-binary transformers[torch] huggingface_hub[hf_xet] -q
%pip install nltk  sqlglot datasets sentencepiece -q
%pip install pybaseball peft -q

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
### Postgres Imports
import psycopg2
from sqlalchemy import create_engine

### Spark Imports
from pyspark.sql import DataFrame
from pyspark.sql.functions import rand, lit, udf
from pyspark.sql.types import StringType

import pandas as pd
import sqlglot
import numpy as np
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction

from transformers import T5Tokenizer, T5ForConditionalGeneration, Trainer, TrainingArguments
from datasets import Dataset

#pybaseball imports
from pybaseball import playerid_reverse_lookup, fielding_stats, pitching_stats

from peft import get_peft_model, LoraConfig, TaskType, PeftModel

import os
import warnings
import re
import random
import torch
from time import time

# Suppress OMP/MKL warning
os.environ["OMP_NUM_THREADS"] = "8"
os.environ["MKL_NUM_THREADS"] = "8"

# Suppress specific warnings
warnings.filterwarnings("ignore", message=".*OMP_NUM_THREADS/MKL_NUM_THREADS unset.*")
warnings.filterwarnings("ignore", message=r".*invalid escape sequence '\\\..*")
warnings.filterwarnings("ignore", message=".*Passing a tuple of `past_key_values` is deprecated.*")

baseline_model_path = "***"
tokenizer_path = "***"

statcast_train_directory = "***"
train_directory = "***"
test_directory = "***"
batter_directory = "***"
pitcher_directory = "***"
baseline_predictions_directory = "***"

#### Get Training data

In [0]:
def get_player_name_from_id(player_id: int) -> str:
    player_info = playerid_reverse_lookup([player_id])
    return f"{player_info['name_first'][0].capitalize()} {player_info['name_last'][0].capitalize()}"

train = spark.read.parquet(statcast_train_directory)

pitchers = spark.read.parquet(pitcher_directory)
pitchers = pitchers.withColumnRenamed("pitcher", "player")
pitchers_udf = udf(get_player_name_from_id, StringType())
pitchers = pitchers.withColumn("name", pitchers_udf(pitchers["player"]))

batters = spark.read.parquet(batter_directory)
batters = batters.withColumnRenamed("batter", "player")
batters_udf = udf(get_player_name_from_id, StringType())
batters = batters.withColumn("name", batters_udf(batters["player"]))

## Postgres

### Postgres Setup

In [0]:
jdbc_url = "***"
rds_endpoint = "***"

connection_properties = {
    "user": "***",
    "password": "***",
    "driver": "***",
    "port": "***"
}

username = connection_properties['user']
password = connection_properties['password']
port = connection_properties['port']
database_name = 'statcast'

engine = create_engine(
    f'postgresql://{username}:{password}@{rds_endpoint}:{port}/{database_name}'
)

def get_conn():
    return psycopg2.connect(
        host=rds_endpoint,
        database=database_name,
        user=username,
        password=password,
        port=port
    )

### Postgres Utility Functions

#### Query Utility Functions

In [0]:
def read_from_rds(query: str) -> DataFrame:
    return spark.read.jdbc(
        url = jdbc_url,
        table = query,
        properties=connection_properties
    )

def get_table_schema(tablename: str):
    query = f"(select column_name, data_type, is_nullable, column_default from information_schema.columns where table_name = '{tablename}' order by ordinal_position) as schema_query"
    return read_from_rds(query)

In [0]:
def get_tables_as_list():
    tables = list_tables().select("tablename").distinct().rdd.flatMap(lambda x: x).collect()
    tables.remove('statcast_batters')
    tables.remove('statcast_pitchers')

    return tables

#### Postgres Metadata

In [0]:
def get_database_table_sizes():
    tables_df = spark.read \
    .format("jdbc") \
    .option("url", jdbc_url) \
    .option("query", "SELECT relname AS table_name, pg_size_pretty(pg_total_relation_size(relid)) AS total_size FROM pg_catalog.pg_statio_user_tables") \
    .option("user", connection_properties.get("user")) \
    .option("password", connection_properties.get("password")) \
    .load()

    tables_df.show(truncate=False)

def list_tables():
    tables_df = spark.read \
    .format("jdbc") \
    .option("url", jdbc_url) \
    .option("query", "SELECT schemaname, tablename FROM pg_catalog.pg_tables WHERE schemaname NOT IN ('pg_catalog', 'information_schema')") \
    .option("user", connection_properties.get("user")) \
    .option("password", connection_properties.get("password")) \
    .load()

    return tables_df

def get_table_schema(tablename: str):
    query = f"(select column_name, data_type, is_nullable, column_default from information_schema.columns where table_name = '{tablename}' order by ordinal_position) as schema_query"
    return read_from_rds(query)

### Split into train and test data

In [0]:
def get_primary_positions():
    results = []
    for year in range(2015, 2025):
        try:
            df = fielding_stats(year)
            # Sum innings by player and position
            grouped = (
                df.groupby(["Name", "Pos"])["Inn"]
                .sum()
                .reset_index()
            )

            # Get primary position per player
            primary_pos = (
                grouped.sort_values("Inn", ascending=False)
                .groupby("Name")
                .first()
                .reset_index()
            )

            primary_pos["year"] = year
            results.append(primary_pos[["Name", "Pos", "year"]])
        except Exception as e:
            print(f"Failed for year {year}: {e}")

        try:
            pitching_df = pitching_stats(year)
            pitching_df = pitching_df[["Name"]].drop_duplicates()
            pitching_df["Pos"] = "P"
            pitching_df["year"] = year
        except Exception as e:
            print(f"Pitching stats failed for {year}: {e}")

        results.append(pitching_df)

    return results

# Combine all years
primary_positions = pd.concat(get_primary_positions(), ignore_index=True)
primary_positions.columns = ["name", "position", "year"]
primary_positions['position'] = primary_positions['position'].apply(lambda x: 'OF' if x in ['LF', 'CF', 'RF'] else x)

In [0]:
def split_train_test(df: DataFrame):
    train, test = df.randomSplit([0.8, 0.2], seed=1234)

    train = train.orderBy(rand(10))
    test = test.orderBy(rand(10))

    return train, test

train, test = split_train_test(train)

train.write.mode("overwrite").parquet(train_directory)
test.write.mode("overwrite").parquet(test_directory)

train_dataset = Dataset.from_pandas(train.toPandas())
test_dataset = Dataset.from_pandas(test.toPandas())

## Models

### Baseline T5 Model

#### Create Tokenizer and Tokenize the Train Data

In [0]:
### Tokenize the dataset
tokenizer = T5Tokenizer.from_pretrained("t5-small")

def tokenize_function(example):
    input_enc = tokenizer(example["nl_query"], max_length=64, truncation=True, padding="max_length")
    target_enc = tokenizer(example["sql_query"], max_length=128, truncation=True, padding="max_length")
    input_enc["labels"] = target_enc["input_ids"]
    return input_enc

def get_tokenized_dataset(dataset):
    return dataset.map(tokenize_function, remove_columns=train_dataset.column_names)

tokenized_dataset = get_tokenized_dataset(train_dataset)

tokenizer_config.json:   0%|          | 0.00/2.32k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.39M [00:00<?, ?B/s]

You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565


Map:   0%|          | 0/18630 [00:00<?, ? examples/s]

#### Train the Model

In [0]:
### Fine-Tune T5 Model
def train_baseline_model():
    model = T5ForConditionalGeneration.from_pretrained("t5-small")

    training_args = TrainingArguments(
        output_dir="/dbfs/tmp/t5_sql_gen_model",
        per_device_train_batch_size=32,
        num_train_epochs=3,
        logging_dir="/dbfs/tmp/logs",
        logging_steps=10,
        save_strategy="no",
        fp16=True,
        ddp_find_unused_parameters=False
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=tokenized_dataset
    )

    trainer.train()

    model.save_pretrained("***")
    tokenizer.save_pretrained("***")

#train_baseline_model()

com.databricks.backend.common.rpc.CommandCancelledException
	at com.databricks.spark.chauffeur.ExecContextState.cancel(ExecContextState.scala:434)
	at com.databricks.spark.chauffeur.ExecutionContextManagerV1.cancelExecution(ExecutionContextManagerV1.scala:464)
	at com.databricks.spark.chauffeur.ChauffeurState.$anonfun$process$1(ChauffeurState.scala:737)
	at com.databricks.logging.UsageLogging.$anonfun$recordOperation$1(UsageLogging.scala:508)
	at com.databricks.logging.UsageLogging.executeThunkAndCaptureResultTags$1(UsageLogging.scala:613)
	at com.databricks.logging.UsageLogging.$anonfun$recordOperationWithResultTags$4(UsageLogging.scala:636)
	at com.databricks.logging.AttributionContextTracing.$anonfun$withAttributionContext$1(AttributionContextTracing.scala:49)
	at com.databricks.logging.AttributionContext$.$anonfun$withValue$1(AttributionContext.scala:293)
	at scala.util.DynamicVariable.withValue(DynamicVariable.scala:62)
	at com.databricks.logging.AttributionContext$.withValue(Attr

#### Load the Saved Model and Tokenizer

In [0]:
model = T5ForConditionalGeneration.from_pretrained(baseline_model_path)
tokenizer = T5Tokenizer.from_pretrained(tokenizer_path)

### Baseline Model with 5 Epochs

In [0]:
### Fine-Tune T5 Model
def fine_tune_five_epoch_t5_model():
    start = time()
    five_epoch_model = T5ForConditionalGeneration.from_pretrained("t5-small")

    training_args = TrainingArguments(
        output_dir="/dbfs/tmp/t5_sql_gen_model",
        per_device_train_batch_size=32,
        num_train_epochs=5,
        logging_dir="/dbfs/tmp/logs",
        logging_steps=10,
        save_strategy="no",
        fp16=True,
        ddp_find_unused_parameters=False
    )

    trainer = Trainer(
        model=five_epoch_model,
        args=training_args,
        train_dataset=tokenized_dataset
    )

    trainer.train()
    wall_time = time() - start
    print("Wall Time For 5 Epoch Model:", wall_time)

    five_epoch_model.save_pretrained("***")
    tokenizer.save_pretrained("****")

fine_tune_five_epoch_t5_model()

Step,Training Loss
10,8.448700
20,4.583500
30,3.222200
40,2.601500
50,2.151000
60,1.771300
70,1.485200
80,1.221400
90,1.076000
100,0.928100


Wall Time For 5 Epoch Model: 14064.021888494492


### Additional Model

In [0]:
def fine_tune_regularized_model():
    start = time()
    five_epoch_model = T5ForConditionalGeneration.from_pretrained("t5-small")

    training_args = TrainingArguments(
        output_dir="/dbfs/tmp/t5_sql_gen_model",
        per_device_train_batch_size=32,
        num_train_epochs=3,
        logging_dir="/dbfs/tmp/logs",
        logging_steps=25,
        save_strategy="no",
        fp16=True,
        ddp_find_unused_parameters=False,
        learning_rate=3e-4,
        warmup_steps=250,
        weight_decay=0.01,
        optim="adafactor"
    )

    trainer = Trainer(
        model=five_epoch_model,
        args=training_args,
        train_dataset=tokenized_dataset
    )

    trainer.train()
    wall_time = time() - start
    print("Wall Time For Regularized Model:", wall_time)

    five_epoch_model.save_pretrained("***")
    tokenizer.save_pretrained("***")

    return five_epoch_model, tokenizer

model, tokenizer = fine_tune_regularized_model()

Step,Training Loss
25,9.785700
50,4.972000
75,2.114700
100,1.081700
125,0.559800
150,0.327400
175,0.240800
200,0.185200
225,0.142600
250,0.123000


Wall Time For Regularized Model: 9396.77792930603


### Model with LoRa

In [0]:
def fine_tune_regularized_model_lora(epochs=3, use_dora=False):
    start = time()

    # Load base model
    model = T5ForConditionalGeneration.from_pretrained("t5-small")

    # Define LoRA config
    lora_config = LoraConfig(
        r=8,
        lora_alpha=32,
        target_modules=["q", "v"],
        lora_dropout=0.1,
        bias="none",
        task_type=TaskType.SEQ_2_SEQ_LM,
        use_dora=use_dora
    )

    # Wrap model with LoRA
    model = get_peft_model(model, lora_config)

    # Training arguments
    training_args = TrainingArguments(
        output_dir="/dbfs/tmp/t5_sql_gen_model_lora",
        per_device_train_batch_size=32,
        num_train_epochs=epochs,
        logging_dir="/dbfs/tmp/logs",
        logging_steps=25,
        save_strategy="no",
        fp16=True,
        ddp_find_unused_parameters=False,
        learning_rate=3e-4,
        warmup_steps=250,
        weight_decay=0.01,
        optim="adafactor"
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=tokenized_dataset
    )

    trainer.train()
    wall_time = time() - start

    model_type = "dora" if use_dora else "lora"

    print(f"Wall Time For {model_type} {epochs} epochs Model:", wall_time)

    # Save the LoRA adapter (not full model)
    model.save_pretrained(f"***")
    tokenizer.save_pretrained("***")

    return model, tokenizer

In [0]:
model, tokenizer = fine_tune_regularized_model_lora()

config.json:   0%|          | 0.00/1.21k [00:00<?, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/242M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

No label_names provided for model class `PeftModelForSeq2SeqLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.
Passing a tuple of `past_key_values` is deprecated and will be removed in Transformers v4.48.0. You should pass an instance of `EncoderDecoderCache` instead, e.g. `past_key_values=EncoderDecoderCache.from_legacy_cache(past_key_values)`.


Step,Training Loss
25,10.661500
50,10.446400
75,9.348800
100,7.037500
125,4.009100
150,2.859000
175,2.241000
200,1.821900
225,1.514700
250,1.227600


Wall Time For LoRA Model: 5866.701997756958


In [0]:
base = T5ForConditionalGeneration.from_pretrained("t5-small")
model = PeftModel.from_pretrained(T5ForConditionalGeneration.from_pretrained("t5-small"), "***")

In [0]:
lora_5_epochs, tokenizer = fine_tune_regularized_model_lora(epochs=5)
lora_5_model = PeftModel.from_pretrained(T5ForConditionalGeneration.from_pretrained("t5-small"), lora_5_epochs)

No label_names provided for model class `PeftModelForSeq2SeqLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.
Passing a tuple of `past_key_values` is deprecated and will be removed in Transformers v4.48.0. You should pass an instance of `EncoderDecoderCache` instead, e.g. `past_key_values=EncoderDecoderCache.from_legacy_cache(past_key_values)`.


Step,Training Loss
25,10.657100
50,10.409700
75,9.238800
100,6.897600
125,3.976400
150,2.851400
175,2.267500
200,1.838100
225,1.529200
250,1.239700


Wall Time For lora 5 epochs Model: 11786.53085231781


In [0]:
dora_3_epochs, tokenizer = fine_tune_regularized_model_lora(epochs=3, use_dora=True)
dora_3_model = PeftModel.from_pretrained(T5ForConditionalGeneration.from_pretrained("t5-small"), dora_3_epochs)

No label_names provided for model class `PeftModelForSeq2SeqLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.


Step,Training Loss
25,10.658600
50,10.419400
75,9.292500
100,7.062100
125,4.144600
150,2.844600
175,2.240000
200,1.826300
225,1.518900
250,1.227700


Wall Time For dora 3 epochs Model: 7151.336558818817


In [0]:
dora_5_epochs, tokenizer = fine_tune_regularized_model_lora(epochs=5, use_dora=True)
dora_5_model = PeftModel.from_pretrained(T5ForConditionalGeneration.from_pretrained("t5-small"), dora_5_epochs)

No label_names provided for model class `PeftModelForSeq2SeqLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.


Step,Training Loss
25,10.657200
50,10.413500
75,9.230400
100,6.943200
125,3.970400
150,2.848000
175,2.270500
200,1.834700
225,1.522900
250,1.232200


Wall Time For dora 5 epochs Model: 11727.510630130768


### Model Testing Utility Functions

In [0]:
def preprocess_query(raw_nl_query):
    """Convert raw NL query to a templated form and extract slot values."""
    slot_values = {}

    # Top/Bottom N
    match_top_bottom = re.search(r"\b(top|bottom)\s+(\d+)", raw_nl_query, flags=re.IGNORECASE)
    if match_top_bottom:
        slot_values["[N]"] = match_top_bottom.group(2)
        raw_nl_query = re.sub(r"\b(top|bottom)\s+\d+\b", r"\1 [N]", raw_nl_query, flags=re.IGNORECASE)

    # Year
    match_year = re.search(r"\b(20[1-2][0-9])\b", raw_nl_query)
    if match_year:
        slot_values["[YEAR]"] = match_year.group(1)
        raw_nl_query = raw_nl_query.replace(match_year.group(1), "[YEAR]")

    # Any number (thresholds)
    match_numeric = re.findall(r"\b\d+(?:\.\d+)?\b", raw_nl_query)
    for num in match_numeric:
        if "[NUM]" not in slot_values.values():
            slot_values["[NUM]"] = num
            raw_nl_query = re.sub(r"\b" + re.escape(num) + r"\b", "[NUM]", raw_nl_query, count=1)

    # Player name
    match_player = re.search(r"\b([A-Z][a-z]+)\s+([A-Z][a-z]+)\b", raw_nl_query)
    if match_player:
        first, last = match_player.groups()
        slot_values["[FIRST]"] = first
        slot_values["[LAST]"] = last
        slot_values["[PLAYER]"] = f"{first} {last}"
        raw_nl_query = raw_nl_query.replace(f"{first} {last}", "[PLAYER]")

    return raw_nl_query, slot_values


def postprocess_sql(sql_template, slot_values):
    """Replace placeholders in SQL with real values from the original query."""
    for slot, value in slot_values.items():
        sql_template = sql_template.replace(slot, str(value))
    return sql_template


def predict_sql(raw_nl_query, model, tokenizer, max_length=128):
    """Preprocess, generate SQL using model, and postprocess the result."""
    # Step 1: Preprocessing
    templated_nl, slot_values = preprocess_query(raw_nl_query)

    # Step 2: Generate SQL from model
    input_ids = tokenizer(templated_nl, return_tensors="pt").input_ids
    output_ids = model.generate(input_ids, max_length=max_length, do_sample=False, num_beams=1)
    sql_template = tokenizer.decode(output_ids[0], skip_special_tokens=True)

    # Step 3: Postprocessing
    final_sql = postprocess_sql(sql_template, slot_values)

    return final_sql


In [0]:
def generate_sql(nl_query):
    input_ids = tokenizer(nl_query, return_tensors="pt").input_ids

    torch.manual_seed(42)
    torch.cuda.manual_seed_all(42)
    random.seed(42)
    np.random.seed(42)
    torch.use_deterministic_algorithms(True)

    output_ids = model.generate(
        input_ids,
        max_length=128,
        num_beams=1,
        no_repeat_ngram_size=2,
        do_sample=False
    )
    return tokenizer.decode(output_ids[0], skip_special_tokens=True)

def normalize_sql(sql):
    # Remove repeated WHERE/ORDER BY
    sql = re.sub(r"(?i)(WHERE\s+.*?)(WHERE\s+)", r"\1", sql)
    sql = re.sub(r"(?i)(ORDER BY\s+.*?)(ORDER BY\s+)", r"\1", sql)

    # Normalize spacing
    sql = sql.strip().replace("  ", " ")

    # Add semicolon if missing
    if not sql.strip().endswith(";"):
        sql += ";"
    return sql

In [0]:
def fill_template_query(templated_nl_query, sql_query, primary_positions):
    slot_values = {}

    # --- Extract table name ---
    match_table = re.search(r"\bFROM\s+([a-zA-Z_]+)", sql_query, flags=re.IGNORECASE)
    table = match_table.group(1) if match_table else ""

    # --- Fill [YEAR] ---
    match_year = re.search(r"\b(20[1-2][0-9])\b", sql_query)
    year = match_year.group(1) if match_year else str(random.randint(2015, 2024))
    slot_values["[YEAR]"] = year

    # --- Fill [N] ---
    if "[N]" in templated_nl_query or "[N]" in sql_query:
        slot_values["[N]"] = str(random.choice([3, 5, 10]))

    # --- Fill [NUM] ---
    if "[NUM]" in templated_nl_query or "[NUM]" in sql_query:
        slot_values["[NUM]"] = str(random.choice([95, 100, 85]))

    # --- Determine allowed positions for [PLAYER] ---
    allowed_positions = None
    if any(x in table for x in ["batter", "oaa", "fielding", "running", "sprint"]) and "batter_pitcher_arsenal" not in table:
        allowed_positions = ["1B", "2B", "3B", "SS", "OF", "C"]
    elif "pitch" in table and "batter_pitcher_arsenal" not in table:
        allowed_positions = ["P"]
    elif "outfield" in table:
        allowed_positions = ["OF"]
    elif "catcher" in table:
        allowed_positions = ["C"]

    if "[PLAYER]" in templated_nl_query and allowed_positions:
        candidates = primary_positions[
            (primary_positions["year"].astype(str) == year) &
            (primary_positions["position"].isin(allowed_positions))
        ]
        if not candidates.empty:
            slot_values["[PLAYER]"] = random.choice(candidates["name"].tolist())

    # --- Apply replacements ---
    final_nl = templated_nl_query
    final_sql = sql_query
    for key, val in slot_values.items():
        final_nl = final_nl.replace(key, val)
        final_sql = final_sql.replace(key, val)

    return final_nl, final_sql

In [0]:
def fill_test_queries(test):
    test_pd = test.toPandas()

    # Apply row-wise filling
    filled = test_pd.apply(
        lambda row: fill_template_query(row["nl_query"], row["sql_query"], primary_positions),
        axis=1,
        result_type="expand"
    )
    filled.columns = ["nl_query", "sql_query"]

    return spark.createDataFrame(filled)

In [0]:
test = fill_test_queries(test)

In [0]:
tokenizer = T5Tokenizer.from_pretrained("t5-small")

model = T5ForConditionalGeneration.from_pretrained("***")

tokenizer_config.json:   0%|          | 0.00/2.32k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.39M [00:00<?, ?B/s]

You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565


### Run Model Testing

In [0]:
def batch_predict_sql(nl_queries, model, tokenizer, batch_size=16, max_length=128, verbose=False):
    model.eval()
    all_final_sql = []

    for i in range(0, len(nl_queries), batch_size):
        batch = nl_queries[i:i+batch_size]
        if verbose:
            print(f"\nProcessing batch {i // batch_size + 1} ({len(batch)} examples)")

        t1 = time()
        templated_nls = []
        slot_values_list = []
        for query in batch:
            templated, slots = preprocess_query(query)
            templated_nls.append(templated)
            slot_values_list.append(slots)
        if verbose:
            print(f"Preprocessing time: {time() - t1:.2f}s")

        t2 = time()
        encodings = tokenizer(
            templated_nls,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=max_length
        )

        if torch.cuda.is_available():
            model = model.to("cuda")
            encodings = {k: v.to("cuda") for k, v in encodings.items()}

        with torch.no_grad():
            output_ids = model.generate(
                **encodings,
                max_length=max_length,
                num_beams=1,
                do_sample=False
            )
        decoded_templates = tokenizer.batch_decode(output_ids, skip_special_tokens=True)
        if verbose:
            print(f"Model generate time: {time() - t2:.2f}s")

        # --- Step 4: Postprocess ---
        t3 = time()
        for sql_template, slots in zip(decoded_templates, slot_values_list):
            final_sql = postprocess_sql(sql_template, slots)
            all_final_sql.append(final_sql)
        if verbose:
            print(f"Postprocessing time: {time() - t3:.2f}s")

    return all_final_sql

In [0]:
def run_model_testing(
    test_data,
    model,
    tokenizer = T5Tokenizer.from_pretrained("t5-small"),
    batch_size=16
):
    test_data = fill_test_queries(test_data)
    test_pd = test_data.select("nl_query", "sql_query").toPandas()

    # run batch predictions
    test_pd["predicted_sql"] = batch_predict_sql(
        test_pd["nl_query"].tolist(),
        model,
        tokenizer,
        batch_size=batch_size,
        verbose=True  # optional timing logs
    )

    return test_pd


In [0]:
five_epochs_predictions = run_model_testing(
    test,
    T5ForConditionalGeneration.from_pretrained("***")
)

regularized_model_predictions = run_model_testing(
    test,
    T5ForConditionalGeneration.from_pretrained("***")
)

lora_predictions = run_model_testing(
    test,
    model
)

lora_5_epochs_predictions = run_model_testing(
    test,
    PeftModel.from_pretrained(
        T5ForConditionalGeneration.from_pretrained("t5-small"),
        "***"
    )
)

dora_3_epochs_predictions = run_model_testing(
    test,
    PeftModel.from_pretrained(
        T5ForConditionalGeneration.from_pretrained("t5-small"),
        "***"
    )
)

dora_5_epochs_predictions = run_model_testing(
    test,
    PeftModel.from_pretrained(
        T5ForConditionalGeneration.from_pretrained("t5-small"),
        "***"
    )
)


Processing batch 1 (16 examples)
Preprocessing time: 0.00s
Model generate time: 2.68s
Postprocessing time: 0.00s

Processing batch 2 (16 examples)
Preprocessing time: 0.00s
Model generate time: 1.43s
Postprocessing time: 0.00s

Processing batch 3 (16 examples)
Preprocessing time: 0.00s
Model generate time: 1.54s
Postprocessing time: 0.00s

Processing batch 4 (16 examples)
Preprocessing time: 0.00s
Model generate time: 1.45s
Postprocessing time: 0.00s

Processing batch 5 (16 examples)
Preprocessing time: 0.00s
Model generate time: 1.27s
Postprocessing time: 0.00s

Processing batch 6 (16 examples)
Preprocessing time: 0.00s
Model generate time: 2.49s
Postprocessing time: 0.00s

Processing batch 7 (16 examples)
Preprocessing time: 0.00s
Model generate time: 1.53s
Postprocessing time: 0.00s

Processing batch 8 (16 examples)
Preprocessing time: 0.00s
Model generate time: 1.86s
Postprocessing time: 0.00s

Processing batch 9 (16 examples)
Preprocessing time: 0.00s
Model generate time: 1.42s
P

In [0]:
# Save as Parquet to DBFS
spark.createDataFrame(test_pd).write.mode("overwrite").parquet(baseline_predictions_directory)

### Metrics of T5 Models on the Test Set

In [0]:
predictions = spark.read.parquet(baseline_predictions_directory)

In [0]:
def compute_bleu(pred, ref):
    smoothie = SmoothingFunction().method4
    pred_tokens = pred.strip().split()
    ref_tokens = ref.strip().split()
    return sentence_bleu([ref_tokens], pred_tokens, smoothing_function=smoothie)

def extract_components(sql):
    try:
        parsed = sqlglot.parse_one(sql)
        select_cols = set(expr.name for expr in parsed.find_all(sqlglot.exp.Column) if expr.alias_or_name)
        where_filters = set(expr.name for expr in parsed.find_all(sqlglot.exp.Condition) if hasattr(expr, 'name'))
        return select_cols | where_filters
    except Exception:
        return set()

def evaluate_sql_predictions(df, pred_col="predicted_sql", ref_col="sql_query"):
    bleu_scores = []
    component_f1s = []

    for i, row in df.iterrows():
        pred = row[pred_col]
        ref = row[ref_col]

        # BLEU Score
        bleu_scores.append(compute_bleu(pred, ref))

        # Component F1
        pred_comps = extract_components(pred)
        ref_comps = extract_components(ref)
        if pred_comps or ref_comps:
            intersection = len(pred_comps & ref_comps)
            precision = intersection / len(pred_comps) if pred_comps else 0
            recall = intersection / len(ref_comps) if ref_comps else 0
            f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0
        else:
            f1 = 1.0
        component_f1s.append(f1)

    return {
        "Average BLEU Score": np.mean(bleu_scores),
        "Component-wise F1": np.mean(component_f1s)
    }

In [0]:
evaluate_sql_predictions(predictions.toPandas())

{'Exact Match Accuracy': 0.39686628031766474,
 'Structural Match Accuracy': 0.0,
 'Average BLEU Score': 0.6807967337642886,
 'Component-wise F1': 0.79925331641096}

In [0]:
evaluate_sql_predictions(five_epochs_predictions)

{'Exact Match Accuracy': 0.4642627173213136,
 'Structural Match Accuracy': 0.0,
 'Average BLEU Score': 0.7595531410293349,
 'Component-wise F1': 0.839284993153467}

In [0]:
evaluate_sql_predictions(regularized_model_predictions)

{'Exact Match Accuracy': 0.5951921013092938,
 'Structural Match Accuracy': 0.0,
 'Average BLEU Score': 0.8537363212941022,
 'Component-wise F1': 0.8692223545361584}

In [0]:
evaluate_sql_predictions(lora_predictions)

{'Exact Match Accuracy': 0.2427559562137798,
 'Structural Match Accuracy': 0.0,
 'Average BLEU Score': 0.558123245731126,
 'Component-wise F1': 0.7500049405607209}

In [0]:
evaluate_sql_predictions(lora_5_epochs_predictions)

{'Exact Match Accuracy': 0.33676754668383774,
 'Structural Match Accuracy': 0.0,
 'Average BLEU Score': 0.6474048658786327,
 'Component-wise F1': 0.7902881425839089}

In [0]:
evaluate_sql_predictions(dora_3_epochs_predictions)

{'Exact Match Accuracy': 0.24018029620090148,
 'Structural Match Accuracy': 0.0,
 'Average BLEU Score': 0.5669658280381614,
 'Component-wise F1': 0.75900382254383}

In [0]:
evaluate_sql_predictions(dora_5_epochs_predictions)

{'Exact Match Accuracy': 0.3208843099377549,
 'Structural Match Accuracy': 0.0,
 'Average BLEU Score': 0.6479653252938211,
 'Component-wise F1': 0.7971413538320892}